# argmin on Colab (free GPU)

Film a room on your iPhone, upload it from Safari, and this notebook turns it into a 3D scene you can
walk around on your phone. **Your Mac only needs this tab open.**

1. **Runtime → Change runtime type → T4 GPU.**
2. Run the cells **top to bottom** (▶). Steps 1–4 install everything (about 10–15 minutes, once per session).
3. The last cell prints a link and a QR code. Open it on your iPhone.

Scenes live only in this session. Open a scene and use *Save* in your browser, or download from the Files panel on the left (`roomsplat/scans`).

In [ ]:
#@title 1. Check the GPU
import subprocess, sys
out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
print(out.stdout.strip() or out.stderr.strip())
assert out.returncode == 0 and out.stdout.strip(), "No GPU. Use Runtime → Change runtime type → T4 GPU, then run again."
print("GPU OK")

### 2. Install COLMAP (GPU) - about 3 min

In [ ]:
%%bash
set -e
cd /content
if [ ! -x colmap-env/bin/colmap ]; then
  curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj bin/micromamba
  ./bin/micromamba create -y -q -p /content/colmap-env -c conda-forge "colmap=4.2.1=cuda*"
fi
/content/colmap-env/bin/colmap help | head -2
which ffmpeg ffprobe

### 3. Install the trainer + compressor - about 4 min

In [ ]:
%%bash
set -e
cd /content
pip install -q uv qrcode
if [ ! -x /content/gs-env/bin/python ]; then uv venv -q /content/gs-env --python 3.10; fi
P=/content/gs-env/bin/python
# PyTorch 2.4 + the PREBUILT gsplat 1.4 GPU code: no compiling (compiling took over an hour on Colab)
uv pip install -q -p $P "numpy<2" torch==2.4.1 torchvision==0.19.1 --index-url https://download.pytorch.org/whl/cu124
uv pip install -q -p $P "gsplat==1.4.0+pt24cu124" --extra-index-url https://docs.gsplat.studio/whl/pt24cu124 --index-strategy unsafe-best-match
[ -d gsplat ] || git clone -q --depth 1 --branch v1.4.0 https://github.com/nerfstudio-project/gsplat
uv pip install -q -p $P viser nerfview==0.0.2 "imageio[ffmpeg]" scipy scikit-learn tqdm "torchmetrics[image]" opencv-python \
  "tyro>=0.8.8" Pillow tensorboard tensorly pyyaml matplotlib \
  "git+https://github.com/rmbrualla/pycolmap@cc7ea4b7301720ac29287dbe450952511b32125e"
# splat-transform shrinks the scene for the phone
if ! command -v splat-transform >/dev/null; then
  curl -fsSL https://deb.nodesource.com/setup_22.x | bash - >/dev/null 2>&1
  apt-get install -y -qq nodejs >/dev/null 2>&1
  npm install -g @playcanvas/splat-transform@3.11.2 >/dev/null 2>&1
fi
splat-transform --version 2>&1 | head -1
$P -c "import torch, gsplat; print('torch', torch.__version__, '| gsplat', gsplat.__version__, '| cuda', torch.cuda.is_available())"

In [ ]:
#@title 4. Quick GPU check - under a minute
# The trainer wants a "fused_ssim" package that has to be compiled. This is a small drop-in written in plain PyTorch.
import os, subprocess, textwrap
os.makedirs("/content/gs-shim", exist_ok=True)
open("/content/gs-shim/fused_ssim.py", "w").write(textwrap.dedent("""
    import torch, torch.nn.functional as F
    def _win(C, dev, dt, size=11, sigma=1.5):
        x = torch.arange(size, device=dev, dtype=dt) - size // 2
        g = torch.exp(-x ** 2 / (2 * sigma ** 2)); g = g / g.sum()
        return (g[:, None] * g[None, :]).expand(C, 1, size, size).contiguous()
    def fused_ssim(img1, img2, padding="same", train=True):
        C = img1.shape[1]; w = _win(C, img1.device, img1.dtype); pad = 5 if padding == "same" else 0
        c = lambda t: F.conv2d(t, w, padding=pad, groups=C)
        mu1, mu2 = c(img1), c(img2)
        s11, s22, s12 = c(img1 * img1) - mu1 ** 2, c(img2 * img2) - mu2 ** 2, c(img1 * img2) - mu1 * mu2
        C1, C2 = 0.01 ** 2, 0.03 ** 2
        return (((2 * mu1 * mu2 + C1) * (2 * s12 + C2)) / ((mu1 ** 2 + mu2 ** 2 + C1) * (s11 + s22 + C2))).mean()
"""))
test = textwrap.dedent("""
    import sys; sys.path.insert(0, "/content/gs-shim")
    import torch
    from fused_ssim import fused_ssim
    from gsplat.rendering import rasterization
    m = torch.randn(100, 3, device="cuda", requires_grad=True); q = torch.nn.functional.normalize(torch.randn(100, 4, device="cuda"), dim=-1)
    s = torch.rand(100, 3, device="cuda") * 0.1 + 0.01; o = torch.rand(100, device="cuda"); c = torch.rand(100, 3, device="cuda")
    K = torch.tensor([[[100., 0, 64], [0, 100., 64], [0, 0, 1]]], device="cuda"); V = torch.eye(4, device="cuda")[None]; V[0, 2, 3] = 5
    img = rasterization(m, q, s, o, c, V, K, 128, 128)[0]
    (img.sum() + fused_ssim(img.permute(0, 3, 1, 2), img.permute(0, 3, 1, 2).detach().flip(3), padding="valid")).backward()
    print("GPU rendering + training math work on", torch.cuda.get_device_name(0))
""")
r = subprocess.run(["/content/gs-env/bin/python", "-c", test], capture_output=True, text=True)
print(r.stdout.strip() or r.stderr[-2500:])
assert r.returncode == 0, "GPU check failed. Open an issue with the error above."

In [ ]:
#@title Prepare folders
import os
os.makedirs('/content/roomsplat/web', exist_ok=True)
os.makedirs('/content/roomsplat/scans', exist_ok=True)

In [ ]:
%%writefile /content/roomsplat/pipeline.py
"""argmin pipeline: phone video -> camera poses -> 3D Gaussian splat -> compact scene file.

  video --ffmpeg--> sharpest frames --COLMAP (global SfM)--> camera poses --trainer--> splat.ply
        --splat-transform--> scene.spz (small enough for a phone)

Trainers:  msplat (Apple Silicon, Metal; default)  |  gsplat (NVIDIA GPU; set RS_TRAINER=gsplat)

Usage:  python pipeline.py scans/<name>/input.mov [--frames 180] [--iters 7000]
Progress is written to scans/<name>/status.json so the web UI can show it.
"""
import argparse, json, os, shutil, subprocess, sys, time

# Colab sets a notebook-only plotting backend that breaks the separate training Python; children inherit this.
os.environ["MPLBACKEND"] = "Agg"
from pathlib import Path

import numpy as np
from PIL import Image, ImageFilter


def log(scan: Path, stage: str, msg: str = "", pct: float | None = None, done=False, error=None):
    s = {"stage": stage, "msg": msg, "pct": pct, "done": done, "error": error, "t": time.time()}
    (scan / "status.json").write_text(json.dumps(s))
    print(f"[{stage}] {msg}", flush=True)


def run(cmd, scan: Path, logfile="log.txt"):
    with open(scan / logfile, "a") as f:
        f.write("\n$ " + " ".join(map(str, cmd)) + "\n")
        f.flush()
        r = subprocess.run(["nice", "-n", "10"] + list(map(str, cmd)), stdout=f, stderr=subprocess.STDOUT)
    if r.returncode != 0:
        raise RuntimeError(f"{Path(str(cmd[0])).name} failed (see {scan/logfile})")


def sharpness(path: Path) -> float:
    g = Image.open(path).convert("L")
    g.thumbnail((480, 480))
    lap = np.asarray(g.filter(ImageFilter.FIND_EDGES), dtype=np.float32)
    return float(lap.var())


def extract_frames(video: Path, scan: Path, target: int, width: int):
    raw = scan / "raw_frames"
    shutil.rmtree(raw, ignore_errors=True); raw.mkdir()
    dur = float(subprocess.check_output(
        ["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "csv=p=0", str(video)]).strip())
    # oversample 3x, then keep the sharpest frame in each window (kills motion blur)
    fps = min(30.0, max(1.0, target * 3 / dur))
    run(["ffmpeg", "-y", "-i", video, "-vf", f"fps={fps:.3f},scale='min({width},iw)':-2",
         "-q:v", "2", raw / "f_%05d.jpg"], scan)
    files = sorted(raw.glob("*.jpg"))
    imgs = scan / "images"
    shutil.rmtree(imgs, ignore_errors=True); imgs.mkdir()
    win = max(1, len(files) // target)
    kept = 0
    for i in range(0, len(files), win):
        group = files[i:i + win]
        best = max(group, key=sharpness)
        shutil.copy(best, imgs / f"{kept:05d}.jpg"); kept += 1
    shutil.rmtree(raw)
    return kept, dur


GPU = os.environ.get("RS_GPU") == "1"      # set in the cloud, where COLMAP has CUDA
THREADS = "-1" if GPU else str(max(2, (os.cpu_count() or 8) // 2))   # on the Mac: half the cores, keeps it cool


def colmap(scan: Path, robust: bool = False):
    """robust=True uses ALIKED features + LightGlue matching: better on blank walls, but slow on CPU."""
    db, imgs, sparse = scan / "colmap.db", scan / "images", scan / "sparse"
    db.unlink(missing_ok=True); shutil.rmtree(sparse, ignore_errors=True); sparse.mkdir()
    log(scan, "features", "Finding feature points in each frame" + (" (robust mode)" if robust else ""), 20)
    feat = (["--FeatureExtraction.type", "ALIKED_N16ROT"] if robust
            else ["--SiftExtraction.max_num_features", "8192" if GPU else "4096"])
    run(["colmap", "feature_extractor", "--database_path", db, "--image_path", imgs,
         "--ImageReader.single_camera", "1", "--ImageReader.camera_model", "OPENCV",
         "--FeatureExtraction.use_gpu", "1" if GPU else "0", "--FeatureExtraction.num_threads", THREADS] + feat, scan)
    log(scan, "matching", "Matching neighbouring frames", 35)
    match = ["--FeatureMatching.type", "ALIKED_LIGHTGLUE"] if robust else []
    run(["colmap", "sequential_matcher", "--database_path", db,
         "--SequentialMatching.overlap", "10" if robust else "20", "--SequentialMatching.quadratic_overlap", "1",
         "--FeatureMatching.use_gpu", "1" if GPU else "0", "--FeatureMatching.num_threads", THREADS] + match, scan)
    log(scan, "poses", "Solving camera positions (global SfM)", 50)
    run(["colmap", "global_mapper", "--database_path", db, "--image_path", imgs, "--output_path", sparse], scan)
    models = sorted([p for p in sparse.iterdir() if p.is_dir()],
                    key=lambda p: (p / "images.bin").stat().st_size if (p / "images.bin").exists() else 0)
    if not models:
        raise RuntimeError("COLMAP could not reconstruct cameras - try a slower, overlapping video")
    best = models[-1]
    log(scan, "undistort", "Undistorting images", 60)
    out = scan / "dataset"
    shutil.rmtree(out, ignore_errors=True)
    run(["colmap", "image_undistorter", "--image_path", imgs, "--input_path", best,
         "--output_path", out, "--output_type", "COLMAP"], scan)
    # msplat expects sparse/0
    s = out / "sparse"
    if (s / "cameras.bin").exists():
        (s / "0").mkdir(exist_ok=True)
        for f in list(s.glob("*.bin")):
            f.rename(s / "0" / f.name)
    n = len(list((out / "images").glob("*")))
    total = len(list(imgs.glob("*.jpg")))
    return out, n, total


def add_depth_points(scan: Path, dataset: Path):
    """Seed the splat with Depth Anything 3 points so blank walls get geometry. Optional + safe:
    if DA3 isn't installed or its depth disagrees with COLMAP, the scene trains as before."""
    root = Path(__file__).resolve().parent
    py = root / ".venv-da3" / "bin" / "python"
    if not py.exists():
        return 0
    npz = scan / "da3_pts.npz"
    npz.unlink(missing_ok=True)
    log(scan, "depth", "Estimating depth for blank surfaces", 62)
    try:
        run([py, root / "da3_init.py", dataset / "images", scan / "sparse_txt", npz], scan)
    except RuntimeError:
        return 0
    if not npz.exists():
        return 0
    d = np.load(npz)
    xyz, rgb = d["xyz"], d["rgb"]
    txt = scan / "sparse_txt"
    ptxt = txt / "points3D.txt"
    ids = [int(l.split()[0]) for l in ptxt.read_text().splitlines() if l and l[0] != "#"]
    base = max(ids + [0]) + 1
    with open(ptxt, "a") as f:
        for i in range(len(xyz)):
            f.write(f"{base+i} {xyz[i,0]:.5f} {xyz[i,1]:.5f} {xyz[i,2]:.5f} {rgb[i,0]} {rgb[i,1]} {rgb[i,2]} 1.0\n")
    run(["colmap", "model_converter", "--input_path", txt,
         "--output_path", dataset / "sparse" / "0", "--output_type", "BIN"], scan)
    return len(xyz)


def train(scan: Path, dataset: Path, iters: int, downscales: int = 2):
    import msplat
    ds = msplat.load_dataset(str(dataset))
    cfg = msplat.TrainingConfig(iterations=iters, num_downscales=downscales)
    cfg.keep_crs = True  # stay in COLMAP coordinates so view.json poses line up
    tr = msplat.GaussianTrainer(ds, cfg)
    t0 = time.time()

    def cb(s):
        log(scan, "training", f"step {s.iteration}/{iters} · {s.splat_count:,} splats · {time.time()-t0:.0f}s",
            62 + 33 * s.iteration / iters)
    tr.train(cb, callback_every=100)
    ply = scan / "splat.ply"
    tr.export_ply(str(ply))
    return ply


def train_gsplat(scan: Path, dataset: Path, iters: int, downscales: int = 2) -> Path:
    """gsplat 1.4 reference trainer (NVIDIA GPU, prebuilt kernels). Keeps COLMAP coordinates so view.json lines up.
    Env: GSPLAT_DIR (repo with examples/), GSPLAT_PYTHON (its python), GSPLAT_SHIM (dir holding a fused_ssim.py)."""
    import re
    gdir = Path(os.environ.get("GSPLAT_DIR", "/content/gsplat"))
    py = os.environ.get("GSPLAT_PYTHON", sys.executable)
    out = scan / "gsplat_out"
    cmd = [py, str(gdir / "examples" / "simple_trainer.py"), "default",
           "--data_dir", str(dataset), "--data_factor", "1", "--result_dir", str(out),
           "--max_steps", str(iters), "--eval_steps", str(iters), "--save_steps", str(iters),
           "--disable_viewer", "--no-normalize-world-space", "--test_every", "20"]
    env = dict(os.environ)
    if os.environ.get("GSPLAT_SHIM"):
        env["PYTHONPATH"] = os.environ["GSPLAT_SHIM"] + os.pathsep + env.get("PYTHONPATH", "")
    t0 = time.time()
    log(scan, "training", "Training the 3D scene on the GPU", 62)
    proc = subprocess.Popen(cmd, cwd=str(gdir / "examples"), env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    tail, last = b"", 0.0
    with open(scan / "log.txt", "ab") as lf:
        while True:
            chunk = proc.stdout.read1(4096)
            if not chunk:
                break
            lf.write(chunk); lf.flush()
            tail = (tail + chunk)[-600:]
            if time.time() - last > 4:
                m = re.findall(rb"(\d+)/(\d+) \[", tail)
                if m:
                    a_, b_ = int(m[-1][0]), int(m[-1][1])
                    if b_ >= iters // 2:
                        log(scan, "training", f"step {a_}/{b_} · {time.time()-t0:.0f}s", 62 + 33 * a_ / max(b_, 1))
                        last = time.time()
    if proc.wait() != 0:
        raise RuntimeError("GPU training failed (see log.txt)")
    ckpts = sorted((out / "ckpts").glob("ckpt_*_rank0.pt"), key=lambda p: int(p.stem.split("_")[1]))
    if not ckpts:
        raise RuntimeError("Training finished but saved no checkpoint")
    final = scan / "splat.ply"
    run([py, Path(__file__).resolve().parent / "export_ckpt.py", ckpts[-1], final], scan)
    shutil.rmtree(out, ignore_errors=True)
    return final


def qvec2rot(q):
    w, x, y, z = q
    return np.array([
        [1 - 2*y*y - 2*z*z, 2*x*y - 2*w*z, 2*x*z + 2*w*y],
        [2*x*y + 2*w*z, 1 - 2*x*x - 2*z*z, 2*y*z - 2*w*x],
        [2*x*z - 2*w*y, 2*y*z + 2*w*x, 1 - 2*x*x - 2*y*y]])


def export_view(scan: Path, dataset: Path):
    """Camera path + gravity 'up' + room centre, so the viewer opens upright, from where you stood."""
    txt = scan / "sparse_txt"; txt.mkdir(exist_ok=True)
    run(["colmap", "model_converter", "--input_path", dataset / "sparse" / "0",
         "--output_path", txt, "--output_type", "TXT"], scan)
    cams = []
    lines = [l for l in (txt / "images.txt").read_text().split("\n") if not l.startswith("#")]
    for l in lines[::2]:
        if not l.strip():
            continue
        p = l.split()
        R = qvec2rot(list(map(float, p[1:5]))); t = np.array(list(map(float, p[5:8])))
        c = -R.T @ t
        cams.append((p[9], c, R.T @ [0, 0, 1], R.T @ [0, -1, 0]))  # COLMAP: +z forward, -y up
    cams.sort(key=lambda x: x[0])
    ups = np.array([c[3] for c in cams]); up = ups.mean(0); up /= np.linalg.norm(up)
    pts = []
    for l in (txt / "points3D.txt").read_text().splitlines():
        if l and not l.startswith("#"):
            pts.append(list(map(float, l.split()[1:4])))
    pts = np.array(pts) if pts else np.array([c[1] for c in cams])
    centre = np.median(pts, 0)
    r = lambda v: [round(float(x), 5) for x in v]
    view = {"up": r(up), "center": r(centre),
            "cams": [{"p": r(c), "f": r(f), "u": r(u)} for _, c, f, u in cams]}
    (scan / "view.json").write_text(json.dumps(view))


def process_video(video: Path, frames=180, width=1600, iters=7000, downscales=2, no_depth=False, trainer=None):
    """The whole pipeline. `trainer(scan, dataset, iters, downscales) -> ply path` defaults to msplat (Mac)."""
    scan = video.parent
    t0 = time.time()
    try:
        log(scan, "frames", "Extracting sharpest frames", 5)
        n, dur = extract_frames(video, scan, frames, width)
        log(scan, "frames", f"{n} frames from {dur:.0f}s of video", 15)
        dataset, reg, n = colmap(scan)
        if reg < 0.85 * n:   # too many frames lost: retry once with the slower, sturdier features
            log(scan, "poses", f"Only {reg}/{n} frames placed. Retrying in robust mode", 20)
            prev = scan / "dataset_prev"
            shutil.rmtree(prev, ignore_errors=True); dataset.rename(prev)
            d2, reg2, n2 = colmap(scan, robust=True)
            if reg2 > reg:
                dataset, reg, n = d2, reg2, n2
                shutil.rmtree(prev, ignore_errors=True)
            else:   # robust mode didn't help: put the first result back
                shutil.rmtree(d2, ignore_errors=True); prev.rename(d2)
                dataset = d2
        log(scan, "poses", f"{reg}/{n} frames placed in 3D", 61)
        export_view(scan, dataset)
        extra = 0 if no_depth else add_depth_points(scan, dataset)
        ply = (trainer or train)(scan, dataset, iters, downscales)
        log(scan, "compress", "Compressing for the phone", 96)
        st = Path(__file__).resolve().parent / "node_modules" / ".bin" / "splat-transform"
        run([st if st.exists() else "splat-transform", "-w", ply, "--spz-version", "3", scan / "scene.spz"], scan)
        meta = {"frames": n, "registered": reg, "depth_points": extra, "seconds": round(time.time() - t0)}
        (scan / "meta.json").write_text(json.dumps(meta))
        log(scan, "done", f"Done in {meta['seconds']}s ({reg}/{n} frames used)", 100, done=True)
    except Exception as e:
        log(scan, "error", str(e), None, error=str(e))
        raise


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("video", type=Path)
    ap.add_argument("--frames", type=int, default=180)
    ap.add_argument("--width", type=int, default=1600)
    ap.add_argument("--iters", type=int, default=7000)
    ap.add_argument("--downscales", type=int, default=2)
    ap.add_argument("--no-depth", action="store_true", help="skip the Depth Anything 3 step")
    a = ap.parse_args()
    try:
        trainer = train_gsplat if os.environ.get("RS_TRAINER") == "gsplat" else None
        process_video(a.video, a.frames, a.width, a.iters, a.downscales, a.no_depth, trainer)
    except Exception:
        sys.exit(1)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/roomsplat/server.py
"""Upload server: a phone sends a video (in small chunks) and it is turned into a 3D splat automatically.

Runs locally (macOS) or on Google Colab. Optional env:  RS_KEY=<secret>  (require ?key=... once per device)
                                                    PORT=8800
"""
import json, os, queue, re, shutil, socket, subprocess, sys, threading, time
from http.server import ThreadingHTTPServer, SimpleHTTPRequestHandler
from pathlib import Path
from urllib.parse import urlparse, parse_qs

ROOT = Path(__file__).resolve().parent
SCANS = ROOT / "scans"; SCANS.mkdir(exist_ok=True)
PORT = int(os.environ.get("PORT", 8800))
KEY = os.environ.get("RS_KEY", "")
jobs: "queue.Queue[tuple[Path, dict]]" = queue.Queue()
uploads: dict = {}          # upload id -> (video path, status path)
lock = threading.Lock()


def worker():
    while True:
        video, opts = jobs.get()
        cmd = [sys.executable, str(ROOT / "pipeline.py"), str(video),
               "--frames", str(opts.get("frames", 180)), "--iters", str(opts.get("iters", 7000))]
        if shutil.which("caffeinate"):      # macOS: keep the Mac awake while processing
            cmd = ["caffeinate", "-is"] + cmd
        with open(video.parent / "log.txt", "a") as f:
            subprocess.run(cmd, stdout=f, stderr=subprocess.STDOUT, cwd=ROOT)


class H(SimpleHTTPRequestHandler):
    def __init__(self, *a, **k):
        super().__init__(*a, directory=str(ROOT), **k)

    def log_message(self, *a):
        pass

    def end_headers(self):
        self.send_header("Cache-Control", "no-store")
        super().end_headers()

    def json(self, obj, code=200):
        b = json.dumps(obj).encode()
        self.send_response(code)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(b)))
        self.end_headers()
        self.wfile.write(b)

    def gate(self) -> bool:
        """True if the request may proceed. With RS_KEY set, a device must open ?key=... once."""
        if not KEY:
            return True
        u = urlparse(self.path)
        if parse_qs(u.query).get("key", [""])[0] == KEY:
            self.send_response(302)
            self.send_header("Set-Cookie", f"rs={KEY}; Max-Age=31536000; Path=/; HttpOnly; Secure; SameSite=Lax")
            self.send_header("Location", u.path or "/")
            self.send_header("Content-Length", "0")
            self.end_headers()
            return False
        if f"rs={KEY}" not in (self.headers.get("Cookie") or ""):
            self.send_response(401)
            msg = b"Open the full link (with ?key=...) once on this device."
            self.send_header("Content-Length", str(len(msg)))
            self.end_headers()
            self.wfile.write(msg)
            return False
        return True

    def do_GET(self):
        if not self.gate():
            return
        u = urlparse(self.path)
        if u.path == "/":
            self.path = "/web/index.html"
        elif u.path == "/view":
            self.path = "/web/view.html"
        elif u.path == "/api/scans":
            out = []
            for d in sorted(SCANS.iterdir(), key=lambda p: p.stat().st_mtime, reverse=True):
                if not d.is_dir():
                    continue
                st = d / "status.json"
                try:
                    s = json.loads(st.read_text())
                except Exception:
                    s = {"stage": "queued", "msg": "Waiting in queue", "pct": 0}
                out.append({"name": d.name, **s, "ready": (d / "scene.spz").exists()})
            return self.json(out)
        elif u.path.startswith("/scans/"):
            parts = u.path.split("/")
            if len(parts) != 4 or parts[3] not in ("scene.spz", "view.json", "splat.ply", "meta.json"):
                return self.send_error(404)
        elif not u.path.startswith("/web/"):
            return self.send_error(404)
        return super().do_GET()

    def do_POST(self):
        if not self.gate():
            return
        u = urlparse(self.path)
        if u.path != "/upload":
            return self.send_error(404)
        q = {k: v[0] for k, v in parse_qs(u.query).items()}
        uid = re.sub(r"[^a-zA-Z0-9]", "", q.get("id", "x"))[:40] or "x"
        part, parts = int(q.get("part", 0)), int(q.get("parts", 1))
        with lock:
            if part == 0 or uid not in uploads:
                name = re.sub(r"[^a-zA-Z0-9_-]+", "-", q.get("name", "room")).strip("-")[:40] or "room"
                name = f"{time.strftime('%m%d-%H%M')}-{name}"
                ext = re.sub(r"[^a-z0-9]", "", q.get("ext", "mov").lower())[:5] or "mov"
                d = SCANS / name; d.mkdir(parents=True, exist_ok=True)
                video = d / f"input.{ext}"
                video.write_bytes(b"")
                uploads[uid] = video
            video = uploads[uid]
        remaining = int(self.headers.get("Content-Length", 0))
        with open(video, "ab") as f:
            while remaining > 0:
                chunk = self.rfile.read(min(1 << 20, remaining))
                if not chunk:
                    break
                f.write(chunk); remaining -= len(chunk)
        if part == parts - 1:
            (video.parent / "status.json").write_text(json.dumps({"stage": "queued", "msg": "Waiting in queue", "pct": 0}))
            jobs.put((video, {"frames": int(q.get("frames", 180)), "iters": int(q.get("iters", 7000))}))
            uploads.pop(uid, None)
        self.json({"ok": True, "name": video.parent.name, "done": part == parts - 1})


def lan_ip():
    s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
    try:
        s.connect(("10.255.255.255", 1)); return s.getsockname()[0]
    except Exception:
        return "127.0.0.1"
    finally:
        s.close()


if __name__ == "__main__":
    threading.Thread(target=worker, daemon=True).start()
    print(f"Open on your iPhone:  http://{lan_ip()}:{PORT}", flush=True)
    ThreadingHTTPServer(("0.0.0.0", PORT), H).serve_forever()

In [ ]:
%%writefile /content/roomsplat/export_ckpt.py
"""gsplat checkpoint -> standard 3DGS .ply (what splat-transform and every splat viewer reads).
Usage: python export_ckpt.py <ckpt.pt> <out.ply>   (run with the gsplat python: needs torch)"""
import sys
import numpy as np
import torch

ck = torch.load(sys.argv[1], map_location="cpu")["splats"]
means = ck["means"].float().numpy()
scales = ck["scales"].float().numpy()                 # log-space, as the PLY format expects
quats = torch.nn.functional.normalize(ck["quats"].float(), dim=-1).numpy()   # w, x, y, z
opac = ck["opacities"].float().numpy()                # logit-space, as the PLY format expects
sh0 = ck["sh0"].float()                               # N, 1, 3
shN = ck["shN"].float()                               # N, K-1, 3
n = len(means)
f_dc = sh0.reshape(n, 3).numpy()
f_rest = shN.transpose(1, 2).reshape(n, -1).numpy()   # channel-major, like the original 3DGS
m = f_rest.shape[1]
data = np.concatenate([means, np.zeros((n, 3), np.float32), f_dc, f_rest, opac[:, None], scales, quats], 1).astype("<f4")
names = ["x", "y", "z", "nx", "ny", "nz", "f_dc_0", "f_dc_1", "f_dc_2"] + [f"f_rest_{i}" for i in range(m)] \
    + ["opacity", "scale_0", "scale_1", "scale_2", "rot_0", "rot_1", "rot_2", "rot_3"]
assert data.shape[1] == len(names)
hdr = "ply\nformat binary_little_endian 1.0\nelement vertex %d\n" % n + "".join(f"property float {k}\n" for k in names) + "end_header\n"
with open(sys.argv[2], "wb") as f:
    f.write(hdr.encode()); f.write(data.tobytes())
print(f"wrote {n:,} splats")

In [ ]:
%%writefile /content/roomsplat/web/index.html
<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1, viewport-fit=cover">
<meta name="color-scheme" content="light dark">
<meta name="apple-mobile-web-app-capable" content="yes">
<meta name="apple-mobile-web-app-title" content="Rooms">
<meta name="theme-color" content="#f2f2f7" media="(prefers-color-scheme: light)">
<meta name="theme-color" content="#000000" media="(prefers-color-scheme: dark)">
<title>Rooms</title>
<link rel="icon" href="data:,">
<style>
  :root {
    --bg:#f2f2f7; --card:#ffffff; --label:#000; --label2:rgba(60,60,67,.6); --sep:rgba(60,60,67,.18);
    --fill:rgba(120,120,128,.12); --blue:#007aff; --green:#34c759; --red:#ff3b30;
  }
  @media (prefers-color-scheme: dark) {
    :root { --bg:#000; --card:#1c1c1e; --label:#fff; --label2:rgba(235,235,245,.6); --sep:rgba(84,84,88,.6);
            --fill:rgba(120,120,128,.24); --blue:#0a84ff; --green:#30d158; --red:#ff453a; }
  }
  * { box-sizing:border-box; -webkit-tap-highlight-color:transparent; }
  html { -webkit-text-size-adjust:100%; }
  body { margin:0; background:var(--bg); color:var(--label);
         font:17px/1.3 -apple-system,BlinkMacSystemFont,"SF Pro Text",system-ui,sans-serif;
         padding:env(safe-area-inset-top) 0 calc(40px + env(safe-area-inset-bottom)); }
  main { max-width:560px; margin:0 auto; padding:0 16px; }
  h1 { font-size:34px; line-height:1.1; font-weight:700; letter-spacing:.37px; margin:56px 0 4px; }
  .sub { color:var(--label2); font-size:15px; margin:0 0 24px; }

  .primary { display:flex; align-items:center; justify-content:center; gap:8px; width:100%;
             background:var(--blue); color:#fff; border:0; border-radius:14px; padding:16px;
             font:600 17px/1 inherit; font-family:inherit; cursor:pointer; transition:opacity .15s, transform .1s; }
  .primary:active { opacity:.7; transform:scale(.99); }
  .primary[disabled] { opacity:.4; }
  .primary svg { width:20px; height:20px; }

  .seg { display:flex; background:var(--fill); border-radius:9px; padding:2px; margin:14px 0 0; }
  .seg button { flex:1; border:0; background:none; color:var(--label); font:500 13px/1 inherit; font-family:inherit;
                padding:8px 0; border-radius:7px; cursor:pointer; }
  .seg button[aria-pressed=true] { background:var(--card); box-shadow:0 1px 3px rgba(0,0,0,.12), 0 0 0 .5px rgba(0,0,0,.04); font-weight:600; }
  @media (prefers-color-scheme: dark) { .seg button[aria-pressed=true] { background:#636366; } }
  .hint { color:var(--label2); font-size:13px; text-align:center; margin:8px 0 0; min-height:17px; }

  h2 { font-size:13px; font-weight:400; color:var(--label2); text-transform:uppercase; letter-spacing:-.08px; margin:32px 16px 6px; }
  .list { background:var(--card); border-radius:12px; overflow:hidden; }
  .row { display:flex; align-items:center; gap:12px; padding:12px 16px; position:relative; color:inherit; text-decoration:none; }
  .row + .row::before { content:""; position:absolute; top:0; left:16px; right:0; height:.5px; background:var(--sep); }
  .row .t { flex:1; min-width:0; }
  .row b { display:block; font-weight:400; overflow:hidden; text-overflow:ellipsis; white-space:nowrap; }
  .row small { display:block; color:var(--label2); font-size:13px; margin-top:2px; }
  .row small.err { color:var(--red); }
  .row .chev { width:8px; height:14px; color:rgba(60,60,67,.3); flex:none; }
  @media (prefers-color-scheme: dark) { .row .chev { color:rgba(235,235,245,.3); } }
  .prog { height:4px; background:var(--fill); border-radius:2px; margin-top:8px; overflow:hidden; }
  .prog i { display:block; height:100%; background:var(--blue); border-radius:2px; transition:width .4s ease; }
  .spin { width:20px; height:20px; border-radius:50%; border:2.5px solid var(--fill); border-top-color:var(--blue); animation:r 1s linear infinite; flex:none; }
  @keyframes r { to { transform:rotate(360deg); } }
  .empty { padding:22px 16px; text-align:center; color:var(--label2); font-size:15px; }

  details { margin:24px 16px 0; color:var(--label2); font-size:14px; }
  summary { cursor:pointer; color:var(--blue); list-style:none; font-size:15px; }
  summary::-webkit-details-marker { display:none; }
  details ul { padding-left:18px; margin:10px 0 0; }
  details li { margin:5px 0; }
</style>
</head>
<body>
<main>
  <h1>Rooms</h1>
  <p class="sub">Film a room. It becomes a 3D scene you can walk through.</p>

  <input id="file" type="file" accept="video/*" hidden>
  <button class="primary" id="pick">
    <svg viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2" stroke-linecap="round"><path d="M12 5v14M5 12h14"/></svg>
    <span id="pickt">Scan a Room</span>
  </button>
  <div class="seg" id="seg" role="group" aria-label="Quality">
    <button data-v="120,4000" aria-pressed="false">Fast</button>
    <button data-v="180,7000" aria-pressed="true">Balanced</button>
    <button data-v="260,15000" aria-pressed="false">Best</button>
  </div>
  <p class="hint" id="hint">About 6 minutes to process</p>

  <h2>Scenes</h2>
  <div class="list" id="list"><div class="empty">No rooms yet</div></div>

  <details>
    <summary>How to film</summary>
    <ul>
      <li>Record in 1080p for 1 to 2 minutes.</li>
      <li>Walk slowly along the walls, facing inward. Then do a second pass facing the walls.</li>
      <li>Keep the lights on and avoid mirrors and fast spins.</li>
      <li>Always keep some edges or furniture in frame, not just a blank wall.</li>
      <li>For full quality, save the video to the Files app first, then pick it from Files. Choosing it from Photos can shrink it to a tiny size.</li>
    </ul>
  </details>
</main>
<script>
const $ = (id) => document.getElementById(id);
const times = { '120,4000': 'About 3 minutes to process', '180,7000': 'About 6 minutes to process', '260,15000': 'About 15 minutes to process' };
let quality = '180,7000';
$('seg').onclick = (e) => {
  const b = e.target.closest('button'); if (!b) return;
  quality = b.dataset.v;
  document.querySelectorAll('#seg button').forEach(x => x.setAttribute('aria-pressed', x === b));
  $('hint').textContent = times[quality];
};
$('pick').onclick = () => $('file').click();
$('file').onchange = async () => {
  const f = $('file').files[0]; if (!f) return;
  const [frames, iters] = quality.split(',');
  const ext = (f.name.split('.').pop() || 'mov').toLowerCase();
  const CH = 40 * 1024 * 1024, parts = Math.max(1, Math.ceil(f.size / CH));
  const id = Math.random().toString(36).slice(2) + Date.now().toString(36);
  $('pick').disabled = true;
  const done = (msg) => { $('pickt').textContent = 'Scan a Room'; $('pick').disabled = false; $('file').value = ''; if (msg) $('hint').textContent = msg; refresh(); };
  const send = (i) => new Promise((res, rej) => {
    const x = new XMLHttpRequest();
    x.open('POST', `/upload?id=${id}&part=${i}&parts=${parts}&name=Room&ext=${ext}&frames=${frames}&iters=${iters}`);
    x.upload.onprogress = (e) => { $('pickt').textContent = `Uploading ${Math.round((i * CH + e.loaded) / f.size * 100)}%`; };
    x.onload = () => x.status === 200 ? res() : rej(new Error('HTTP ' + x.status));
    x.onerror = () => rej(new Error('network'));
    x.send(f.slice(i * CH, Math.min(f.size, (i + 1) * CH)));
  });
  try {
    for (let i = 0; i < parts; i++) {
      for (let t = 0; ; t++) { try { await send(i); break; } catch (e) { if (t >= 3) throw e; await new Promise(r => setTimeout(r, 1500)); } }
    }
    done(f.size < 4e6 ? 'Small file. For sharper results, pick the video from the Files app.' : 'Processing');
  } catch (e) { done('Upload failed. Check your connection and try again.'); }
};
const chev = '<svg class="chev" viewBox="0 0 8 14" fill="none" stroke="currentColor" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><path d="M1 1l6 6-6 6"/></svg>';
const esc = (s) => String(s).replace(/[&<>"]/g, c => ({'&':'&amp;','<':'&lt;','>':'&gt;','"':'&quot;'}[c]));
const pretty = (n) => n.replace(/^\d{4}-\d{4}-/, '').replace(/[-_]/g, ' ');
const when = (n) => { const m = n.match(/^(\d\d)(\d\d)-(\d\d)(\d\d)/); return m ? `${+m[1]}/${+m[2]}, ${m[3]}:${m[4]}` : ''; };
async function refresh() {
  try {
    const scans = await (await fetch('/api/scans')).json();
    if (!scans.length) return;
    $('list').innerHTML = scans.map(s => {
      const title = esc(pretty(s.name)), sub = esc(s.error ? s.msg : (s.ready ? (when(s.name) || 'Ready') : (s.msg || s.stage)));
      if (s.ready) return `<a class="row" href="/view?scan=${encodeURIComponent(s.name)}"><div class="t"><b>${title}</b><small>${sub}</small></div>${chev}</a>`;
      if (s.error) return `<div class="row"><div class="t"><b>${title}</b><small class="err">${sub}</small></div></div>`;
      return `<div class="row"><div class="t"><b>${title}</b><small>${sub}</small><div class="prog"><i style="width:${s.pct || 3}%"></i></div></div><div class="spin"></div></div>`;
    }).join('');
  } catch {}
}
refresh(); setInterval(refresh, 2000);
</script>
</body>
</html>

In [ ]:
%%writefile /content/roomsplat/web/view.html
<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1, maximum-scale=1, viewport-fit=cover">
<title>Room Viewer</title>
<link rel="icon" href="data:,">
<style>
  :root { --glass:rgba(40,40,44,.55); --hair:rgba(255,255,255,.14); }
  html, body { margin:0; height:100%; background:#000; overflow:hidden; touch-action:none; font:15px -apple-system,BlinkMacSystemFont,"SF Pro Text",system-ui,sans-serif; color:#fff; -webkit-user-select:none; user-select:none; }
  canvas { display:block; }
  .glass { background:var(--glass); backdrop-filter:blur(24px) saturate(180%); -webkit-backdrop-filter:blur(24px) saturate(180%); border:.5px solid var(--hair); }
  #back { position:fixed; top:max(12px,env(safe-area-inset-top)); left:12px; width:44px; height:44px; border-radius:50%; display:grid; place-items:center; color:#fff; text-decoration:none; }
  #dock { position:fixed; left:50%; transform:translateX(-50%); bottom:max(18px,env(safe-area-inset-bottom)); display:flex; gap:2px; padding:6px; border-radius:30px; }
  #dock button { width:52px; height:48px; border:0; background:none; color:#fff; border-radius:24px; display:grid; place-items:center; cursor:pointer; transition:background .15s; }
  #dock button:active { background:rgba(255,255,255,.18); }
  #dock button.on { background:#fff; color:#000; }
  svg { width:22px; height:22px; fill:none; stroke:currentColor; stroke-width:1.8; stroke-linecap:round; stroke-linejoin:round; }
  #msg { position:fixed; top:50%; left:0; right:0; text-align:center; transform:translateY(-50%); color:rgba(235,235,245,.6); }
  #hint { position:fixed; top:max(22px,env(safe-area-inset-top)); left:64px; right:64px; text-align:center; font-size:13px; color:rgba(235,235,245,.7); text-shadow:0 1px 4px rgba(0,0,0,.6); transition:opacity .5s; }
</style>
<script type="importmap">
{ "imports": {
  "three": "https://cdn.jsdelivr.net/npm/three@0.185.0/build/three.module.js",
  "three/addons/": "https://cdn.jsdelivr.net/npm/three@0.185.0/examples/jsm/",
  "@sparkjsdev/spark": "https://cdn.jsdelivr.net/npm/@sparkjsdev/spark@2.4.0/dist/spark.module.js"
} }
</script>
</head>
<body>
<div id="msg">Loading scene…</div>
<div id="hint">Drag to look around. Pinch to zoom.</div>
<a id="back" class="glass" href="/" aria-label="Back"><svg viewBox="0 0 24 24"><path d="M15 5l-7 7 7 7"/></svg></a>
<div id="dock" class="glass">
  <button id="tour" aria-label="Tour"><svg viewBox="0 0 24 24"><path d="M8 5.5v13l11-6.5z" fill="currentColor"/></svg></button>
  <button id="reset" aria-label="Recenter"><svg viewBox="0 0 24 24"><path d="M4 9V6a2 2 0 012-2h3M15 4h3a2 2 0 012 2v3M20 15v3a2 2 0 01-2 2h-3M9 20H6a2 2 0 01-2-2v-3"/><circle cx="12" cy="12" r="2.5"/></svg></button>
  <button id="flip" aria-label="Flip upside down"><svg viewBox="0 0 24 24"><path d="M7 4v16M7 20l-3-3M7 20l3-3M17 20V4M17 4l-3 3M17 4l3 3"/></svg></button>
  <button id="rot" aria-label="Rotate"><svg viewBox="0 0 24 24"><path d="M20 12a8 8 0 11-2.6-5.9M20 4v4h-4"/></svg></button>
</div>
<script type="module">
import * as THREE from 'three';
import { OrbitControls } from 'three/addons/controls/OrbitControls.js';
import { SparkRenderer, SplatMesh } from '@sparkjsdev/spark';

const scanName = new URLSearchParams(location.search).get('scan');
const base = `/scans/${encodeURIComponent(scanName)}/`;
const $ = (id) => document.getElementById(id);
const PLAY = $('tour').innerHTML;
const STOP = '<svg viewBox="0 0 24 24"><rect x="6.5" y="6.5" width="11" height="11" rx="2" fill="currentColor"/></svg>';

const renderer = new THREE.WebGLRenderer({ antialias: false });
renderer.setPixelRatio(Math.min(devicePixelRatio, 2));
renderer.setSize(innerWidth, innerHeight);
document.body.prepend(renderer.domElement);

const scene = new THREE.Scene();
const camera = new THREE.PerspectiveCamera(65, innerWidth / innerHeight, 0.01, 1000);
scene.add(new SparkRenderer({ renderer }));

// fix = auto "up" from the capture cameras; user = manual flip/rotate on top
const user = new THREE.Group(); scene.add(user);
const fix = new THREE.Group(); user.add(fix);
const splat = new SplatMesh({ url: base + (new URLSearchParams(location.search).get('file') || 'scene.spz') });
fix.add(splat);

const controls = new OrbitControls(camera, renderer.domElement);
controls.enableDamping = true; controls.screenSpacePanning = true; controls.rotateSpeed = 0.6;

let view = null;
try { const r = await fetch(base + 'view.json'); if (r.ok) view = await r.json(); } catch {}

const key = 'orient:' + scanName;
let orient = { flip: !view, rot: 0 };   // old scans without view.json are usually upside down
try { orient = JSON.parse(localStorage.getItem(key)) || orient; } catch {}
function applyOrient() {
  user.rotation.set(orient.flip ? Math.PI : 0, orient.rot * Math.PI / 2, 0);
  user.updateMatrixWorld(true);
  try { localStorage.setItem(key, JSON.stringify(orient)); } catch {}
}
if (view) fix.quaternion.setFromUnitVectors(new THREE.Vector3(...view.up), new THREE.Vector3(0, 1, 0));
applyOrient();

const toWorld = (a) => new THREE.Vector3(...a).applyMatrix4(fix.matrixWorld);
const dirWorld = (a) => new THREE.Vector3(...a).transformDirection(fix.matrixWorld);

function resetView() {
  stopTour();
  if (view) {
    const c0 = view.cams[0];
    const p = toWorld(c0.p), target = toWorld(view.center);
    camera.position.copy(p); controls.target.copy(target);
  } else {
    const box = splat.getBoundingBox(true).applyMatrix4(splat.matrixWorld);
    const c = box.getCenter(new THREE.Vector3());
    const r = Math.min(box.getSize(new THREE.Vector3()).length() * 0.25, 4);
    camera.position.copy(c).add(new THREE.Vector3(0, r * 0.4, r)); controls.target.copy(c);
  }
  camera.up.set(0, 1, 0); controls.update();
}

// Tour: fly smoothly along the path you filmed
let tour = null;
function startTour() {
  if (!view || view.cams.length < 2) return;
  const pts = view.cams.map(c => toWorld(c.p));
  const look = view.cams.map(c => toWorld(c.p).add(dirWorld(c.f)));
  tour = { pos: new THREE.CatmullRomCurve3(pts), look: new THREE.CatmullRomCurve3(look), t: 0,
           dur: Math.max(12, view.cams.length * 0.35) };
  controls.enabled = false; $('tour').classList.add('on'); $('tour').innerHTML = STOP;
}
function stopTour() {
  if (!tour) return;
  tour = null; controls.enabled = true;
  const fwd = new THREE.Vector3(); camera.getWorldDirection(fwd);
  controls.target.copy(camera.position).addScaledVector(fwd, 1.0); controls.update();
  $('tour').classList.remove('on'); $('tour').innerHTML = PLAY;
}
$('tour').onclick = () => tour ? stopTour() : startTour();
$('reset').onclick = resetView;
$('flip').onclick = () => { orient.flip = !orient.flip; applyOrient(); resetView(); };
$('rot').onclick = () => { orient.rot = (orient.rot + 1) % 4; applyOrient(); resetView(); };
if (!view) $('tour').hidden = true;
renderer.domElement.addEventListener('pointerdown', () => { $('hint').style.opacity = 0; }, { once: true });
setTimeout(() => { $('hint').style.opacity = 0; }, 6000);

splat.initialized.then(() => { $('msg').remove(); resetView(); })
  .catch(e => { $('msg').textContent = 'Could not load scene: ' + e; });

addEventListener('resize', () => {
  camera.aspect = innerWidth / innerHeight; camera.updateProjectionMatrix();
  renderer.setSize(innerWidth, innerHeight);
});
const clock = new THREE.Clock();
renderer.setAnimationLoop(() => {
  const dt = clock.getDelta();
  if (tour) {
    tour.t += dt / tour.dur;
    if (tour.t >= 1) stopTour();
    else {
      const u = tour.t;
      camera.position.copy(tour.pos.getPointAt(u));
      camera.up.set(0, 1, 0);
      camera.lookAt(tour.look.getPointAt(u));
    }
  } else controls.update();
  renderer.render(scene, camera);
});
</script>
</body>
</html>

*(The five cells above just write the app's files. Run them, nothing to change.)*

In [ ]:
#@title 5. (Optional) Self-test on a sample scene
# OPTIONAL: test the whole GPU pipeline on a sample (a stuffed fox, 50 photos). Takes about 5 minutes.
import os, subprocess, sys
os.environ.update(RS_GPU="1", RS_TRAINER="gsplat", GSPLAT_DIR="/content/gsplat", GSPLAT_PYTHON="/content/gs-env/bin/python", GSPLAT_SHIM="/content/gs-shim",
                  PATH="/content/colmap-env/bin:/usr/local/cuda/bin:" + os.environ["PATH"])
os.chdir("/content")
if not os.path.exists("ngp"):
    subprocess.run("git clone -q --depth 1 --filter=blob:none --sparse https://github.com/NVlabs/instant-ngp ngp && cd ngp && git sparse-checkout set data/nerf/fox", shell=True, check=True)
os.makedirs("/content/roomsplat/scans/sample-fox", exist_ok=True)
subprocess.run("ffmpeg -y -v error -framerate 5 -pattern_type glob -i '/content/ngp/data/nerf/fox/images/*.jpg' -c:v libx264 -pix_fmt yuv420p "
               "-vf 'scale=trunc(iw/2)*2:trunc(ih/2)*2' /content/roomsplat/scans/sample-fox/input.mp4", shell=True, check=True)
r = subprocess.run([sys.executable, "pipeline.py", "scans/sample-fox/input.mp4", "--frames", "50", "--iters", "7000"], cwd="/content/roomsplat")
print("exit", r.returncode)
print(open("/content/roomsplat/scans/sample-fox/status.json").read())
if r.returncode:
    print(open("/content/roomsplat/scans/sample-fox/log.txt").read()[-2500:])

In [ ]:
#@title 6. Start argmin (open the link on your iPhone)
import os, re, secrets, subprocess, sys, time, urllib.request, stat
from IPython.display import display

key = open("/content/rs_key").read() if os.path.exists("/content/rs_key") else secrets.token_urlsafe(16)
open("/content/rs_key", "w").write(key)
os.environ.update(RS_GPU="1", RS_TRAINER="gsplat", GSPLAT_DIR="/content/gsplat", GSPLAT_PYTHON="/content/gs-env/bin/python", GSPLAT_SHIM="/content/gs-shim", RS_KEY=key, PORT="8800",
                  PATH="/content/colmap-env/bin:/usr/local/cuda/bin:" + os.environ["PATH"])
subprocess.run("pkill -f '[r]oomsplat/server.py'; pkill cloudflared", shell=True)
subprocess.Popen([sys.executable, "/content/roomsplat/server.py"], cwd="/content/roomsplat",
                 stdout=open("/content/server.log", "w"), stderr=subprocess.STDOUT)

cf = "/content/cloudflared"
if not os.path.exists(cf):
    urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", cf)
    os.chmod(cf, os.stat(cf).st_mode | stat.S_IEXEC)
tun = subprocess.Popen([cf, "tunnel", "--url", "http://localhost:8800", "--no-autoupdate"],
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url, ready, t0 = None, False, time.time()
while time.time() - t0 < 90 and not (url and ready):      # the address exists before it is connected: wait for both
    line = tun.stdout.readline()
    if not line:
        break
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m:
        url = m.group(0)
    if "Registered tunnel connection" in line:
        ready = True
assert url, "Could not open a public link. Run this cell again."
import threading, urllib.error
threading.Thread(target=lambda: [None for _ in tun.stdout], daemon=True).start()   # keep the tunnel's log pipe drained
for _ in range(30):      # prove the link works from outside before showing it (401 = the server answered)
    try:
        urllib.request.urlopen(url, timeout=5); break
    except urllib.error.HTTPError:
        break
    except Exception:
        time.sleep(2)
else:
    print("Warning: the link is not reachable yet. Wait a minute and try it; if it still fails run this cell again.")
link = f"{url}/?key={key}"

import qrcode
display(qrcode.make(link).get_image().resize((260, 260)))
print("\nOpen this on your iPhone (scan the QR or type it):\n\n" + link + "\n")
print("The server keeps running in the background. Keep this Colab tab open (it can sleep if closed).")

In [ ]:
#@title RESET: stop everything, clear all scenes, start fresh (use when stuck)
import os, shutil, subprocess, time
# 1) kill every leftover process: server, running job, COLMAP, trainer, tunnel
subprocess.run("pkill -9 -f '[r]oomsplat/server.py'; pkill -9 -f '[r]oomsplat/pipeline.py'; "
               "pkill -9 -f '[s]imple_trainer.py'; pkill -9 colmap; pkill -9 splat-transform; pkill -9 cloudflared", shell=True)
time.sleep(2)
assert os.path.exists("/content/roomsplat/server.py"), "App files are missing. Run steps 1-4 and the file cells first."
# 2) wipe all scenes (this also empties the queue)
shutil.rmtree("/content/roomsplat/scans", ignore_errors=True)
os.makedirs("/content/roomsplat/scans", exist_ok=True)
os.environ["MPLBACKEND"] = "Agg"
print("Cleaned up. Starting fresh...")
import os, re, secrets, subprocess, sys, time, urllib.request, stat
from IPython.display import display

key = open("/content/rs_key").read() if os.path.exists("/content/rs_key") else secrets.token_urlsafe(16)
open("/content/rs_key", "w").write(key)
os.environ.update(RS_GPU="1", RS_TRAINER="gsplat", GSPLAT_DIR="/content/gsplat", GSPLAT_PYTHON="/content/gs-env/bin/python", GSPLAT_SHIM="/content/gs-shim", RS_KEY=key, PORT="8800",
                  PATH="/content/colmap-env/bin:/usr/local/cuda/bin:" + os.environ["PATH"])
subprocess.run("pkill -f '[r]oomsplat/server.py'; pkill cloudflared", shell=True)
subprocess.Popen([sys.executable, "/content/roomsplat/server.py"], cwd="/content/roomsplat",
                 stdout=open("/content/server.log", "w"), stderr=subprocess.STDOUT)

cf = "/content/cloudflared"
if not os.path.exists(cf):
    urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", cf)
    os.chmod(cf, os.stat(cf).st_mode | stat.S_IEXEC)
tun = subprocess.Popen([cf, "tunnel", "--url", "http://localhost:8800", "--no-autoupdate"],
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url, ready, t0 = None, False, time.time()
while time.time() - t0 < 90 and not (url and ready):      # the address exists before it is connected: wait for both
    line = tun.stdout.readline()
    if not line:
        break
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m:
        url = m.group(0)
    if "Registered tunnel connection" in line:
        ready = True
assert url, "Could not open a public link. Run this cell again."
import threading, urllib.error
threading.Thread(target=lambda: [None for _ in tun.stdout], daemon=True).start()   # keep the tunnel's log pipe drained
for _ in range(30):      # prove the link works from outside before showing it (401 = the server answered)
    try:
        urllib.request.urlopen(url, timeout=5); break
    except urllib.error.HTTPError:
        break
    except Exception:
        time.sleep(2)
else:
    print("Warning: the link is not reachable yet. Wait a minute and try it; if it still fails run this cell again.")
link = f"{url}/?key={key}"

import qrcode
display(qrcode.make(link).get_image().resize((260, 260)))
print("\nOpen this on your iPhone (scan the QR or type it):\n\n" + link + "\n")
print("The server keeps running in the background. Keep this Colab tab open (it can sleep if closed).")

In [ ]:
#@title 7. Cancel the scene that is running now (the next queued one starts automatically)
import os, json, time, psutil

killed = []
for p in psutil.process_iter(["pid", "cmdline"]):
    cmd = p.info["cmdline"] or []
    if any(c.endswith("roomsplat/pipeline.py") for c in cmd):      # the job that is running now
        scan = next((c for c in cmd if "/scans/" in c or c.startswith("scans/")), None)
        for ch in p.children(recursive=True):                      # stop COLMAP / the trainer first
            try: ch.kill()
            except psutil.Error: pass
        try: p.kill()
        except psutil.Error: pass
        if scan:
            d = os.path.dirname(os.path.join("/content/roomsplat", scan) if not scan.startswith("/") else scan)
            open(os.path.join(d, "status.json"), "w").write(json.dumps(
                {"stage": "error", "msg": "Cancelled", "pct": None, "done": False, "error": "Cancelled", "t": time.time()}))
            killed.append(os.path.basename(d))
print("Cancelled:", killed if killed else "nothing was running. The next queued scene starts by itself.")

In [ ]:
#@title 8. Show the status of all scenes
import pathlib, json
for p in sorted(pathlib.Path("/content/roomsplat/scans").glob("*/status.json"), key=lambda p: p.stat().st_mtime):
    try: print(p.parent.name.ljust(28), json.loads(p.read_text()).get("msg"))
    except Exception: print(p.parent.name)